# Part 1: Image Classification Representation Learning

In this notebook, we'll train a CNN to classify images from CIFAR-10. We explicitly isolate the latent embeddings dynamically across Epoch 0 through 10, combining them into a single comprehensive data structure for visualization.


In [1]:
# Install required dependencies
!pip install torch torchvision pandas numpy scikit-learn umap-learn bokeh

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import pandas as pd
import numpy as np
import os
import umap
from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.transform import factor_cmap
from bokeh.palettes import Category10
from bokeh.layouts import gridplot

output_notebook()


/Users/adityasengupta/miniconda3/envs/juliaenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading BokehJS ...

## 1. Load the CIFAR-10 Dataset


In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
# Subset for speed during lab (5000 images)
subset_indices = torch.randperm(len(trainset))[:5000]
trainset = torch.utils.data.Subset(trainset, subset_indices)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=False)
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')
print(f"Loaded {len(trainset)} images")


100%|██████████| 170M/170M [01:58<00:00, 1.44MB/s] 


Loaded 5000 images


## 2. Model Architecture with Representation Hook


In [4]:
# We define a simple Convolutional Neural Network (CNN) for image tasks.
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, 5)
        self.fc1 = nn.Linear(32 * 5 * 5, 128)
        self.fc2 = nn.Linear(128, 10)
        self.embeddings = None # This will act as our 'hook' to save the latent features

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = torch.relu(self.fc1(x))
        self.embeddings = x.detach().cpu().numpy()
        x = self.fc2(x)
        return x

model = SimpleCNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.003)


## 3. Train & Capture Representations over 11 States
We define a helper function to suck the vectors out of the network for the entire dataset sequentially. We run it immediately (Epoch 0) and then after each training step.


In [5]:
epochs = 10
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

embeddings_per_epoch = []
labels_list = []

def extract_embeddings():
    model.eval()
    all_embs = []
    all_lbls = []
    with torch.no_grad():
        for data in trainloader:
            inputs, labels = data[0].to(device), data[1]
            outputs = model(inputs)
            all_embs.append(model.embeddings)
            all_lbls.extend(labels.numpy())
    model.train()
    return np.vstack(all_embs), all_lbls

# Epoch 0 (Initialization)
embs, lbls = extract_embeddings()
embeddings_per_epoch.append(embs)
labels_list = lbls

print("Training...")
# ---- Main Training Loop ----
# We iterate over the dataset multiple times (epochs)
for epoch in range(epochs):
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        inputs, labels = data[0].to(device), data[1].to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    print(f"Epoch [{(epoch + 1):2d}/{epochs}] loss: {running_loss / len(trainloader):.3f}")
    
    embs, _ = extract_embeddings()
    embeddings_per_epoch.append(embs)


Training...
Epoch [ 1/10] loss: 1.966
Epoch [ 2/10] loss: 1.643
Epoch [ 3/10] loss: 1.484
Epoch [ 4/10] loss: 1.373
Epoch [ 5/10] loss: 1.274
Epoch [ 6/10] loss: 1.197
Epoch [ 7/10] loss: 1.116
Epoch [ 8/10] loss: 1.071
Epoch [ 9/10] loss: 1.011
Epoch [10/10] loss: 0.902


## 4. Export the Unified CSV for Mantis
We combine all the representation values into a wide dataframe tracking the evolution per row.


In [6]:
label_names = [classes[lbl] for lbl in labels_list]
df = pd.DataFrame({
    'title': [f"Image_{i}" for i in range(len(label_names))],
    'categoric': label_names
})

for i, embs in enumerate(embeddings_per_epoch):
    df[f'embedding_epoch_{i}'] = embs.tolist()

csv_name = 'image_analysis_mantis.csv'
df.to_csv(csv_name, index=False)
print(f"Saved {csv_name}!")


Saved image_analysis_mantis.csv!


## 5. Bokeh Grid Visualization
A panel displaying all epochs sequentially.


In [7]:
from bokeh.io import output_notebook
output_notebook()
plots = []
reducer = umap.UMAP(n_neighbors=25, min_dist=0.1, random_state=42)

df['color'] = df['categoric'].map({cls: color for cls, color in zip(classes, Category10[10])})

print("Generating UMAP plots. This will take a moment...")
for i, embs in enumerate(embeddings_per_epoch):
    umap_coords = reducer.fit_transform(embs)
    
    plot_df = pd.DataFrame({
        'coordinate1': umap_coords[:, 0],
        'coordinate2': umap_coords[:, 1],
        'color': df['color'],
        'categoric': df['categoric'],
        'title': df['title']
    })
    
    source = ColumnDataSource(plot_df)
    p = figure(title=f"Epoch {i}", width=300, height=300)
    p.circle('coordinate1', 'coordinate2', size=3, color='color', legend_group='categoric', alpha=0.6, source=source)
    p.axis.visible = False
    if i != 0:
        p.legend.visible = False
        
    hover = HoverTool(tooltips=[("Category", "@categoric"), ("Image ID", "@title")])
    p.add_tools(hover)
    plots.append(p)

show(gridplot(plots, ncols=3))


Loading BokehJS ...

Generating UMAP plots. This will take a moment...


/Users/adityasengupta/miniconda3/envs/juliaenv/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
